# AUC Reports

This notebook prepares vector-format performance tables for the Results section.

## Section 1. Final-selection Performance Heatmap

This section summarizes the final selected models for clinical variables, conventional radiomics, habitat radiomics, DL-derived features, and fusion models across the train, internal test, and external test cohorts.

In [1]:
# ============================================================
# 1. Imports and settings
# ============================================================

import json
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize

# Use Times New Roman consistently for paper figures.
# In this WSL/container environment, the Windows Times fonts are available under /host/c/Windows/Fonts.
times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

# Make exported PDF/EPS text editable in vector software when possible.
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']

model_root = Path('/host/d/projects/Habitats/models/Prognosis')
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)

figure_pdf_path = results_out_dir / 'performance_heatmap.pdf'
figure_eps_path = results_out_dir / 'performance_heatmap.eps'

# The display names here are intentionally paper-facing labels.
# Folder paths point to each model stream's final_selection output.
model_sources = [
    {
        'model': 'Clinical',
        'kind': 'final_selection',
        'folder': model_root / 'clinical' / 'final_selections',
    },
    {
        'model': 'C-radiomics',
        'kind': 'final_selection',
        'folder': model_root / 'whole_image' / 'final_selections',
    },
    {
        'model': 'H-radiomics',
        'kind': 'final_selection',
        'folder': model_root / 'habitats_avg' / 'final_selections',
    },
    {
        'model': 'DL_3D',
        'kind': 'final_selection',
        'folder': model_root / 'dl_3d_ml_all' / 'final_selections',
    },
    {
        'model': 'fusion_soft_vote',
        'kind': 'soft_vote',
        'metrics_path': model_root / 'fusion' / 'soft_vote_metrics.xlsx',
        'algorithm': 'N/A',
    },
    {
        'model': 'fusion_stacking',
        'kind': 'final_selection',
        'folder': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF',
        'algorithm': None,  # If None, the code reads the CV manifest; fallback is folder name.
    },
]

cohort_specs = [
    {
        'display': 'train',
        'final_selection_file': 'cv_final_selection_metrics.xlsx',
        'soft_vote_dataset': 'cv',
    },
    {
        'display': 'internal test',
        'final_selection_file': 'internal_test_final_selection_metrics.xlsx',
        'soft_vote_dataset': 'internal_test',
    },
    {
        'display': 'external test',
        'final_selection_file': 'external_test_final_selection_metrics.xlsx',
        'soft_vote_dataset': 'external_test',
    },
]

metric_cols = ['auc', 'accuracy', 'sensitivity', 'specificity']
metric_display = {
    'auc': 'AUC',
    'accuracy': 'Accuracy',
    'sensitivity': 'Sensitivity',
    'specificity': 'Specificity',
}

print('Results output directory:', results_out_dir)
print('PDF:', figure_pdf_path)
print('EPS:', figure_eps_path)

def save_pdf_and_ppt_safe_svg(fig, pdf_path, **kwargs):
    """Save the normal PDF plus a PPT-friendly SVG copy.

    The SVG suffix is `_ppt_safe.svg`, so original PDF manuscript outputs stay unchanged.
    """
    pdf_path = Path(pdf_path)
    fig.savefig(pdf_path, **kwargs)
    svg_path = pdf_path.with_name(pdf_path.stem + '_ppt_safe.svg')
    fig.savefig(svg_path, format='svg', **kwargs)
    return svg_path


Results output directory: /host/d/projects/Habitats/results
PDF: /host/d/projects/Habitats/results/performance_heatmap.pdf
EPS: /host/d/projects/Habitats/results/performance_heatmap.eps


In [2]:
# ============================================================
# 2. Helper functions
# ============================================================

def read_json_if_exists(path):
    path = Path(path)
    if not path.is_file():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def normalize_algorithm_name(name):
    """Make algorithm labels consistent for the paper table."""
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ''

    text = str(name).strip()
    lookup = {
        'xgboost': 'XGBoost',
        'xgboost': 'XGBoost',
        'xgboosh': 'XGBoost',
        'svm': 'SVM',
        'lr': 'LR',
        'rf': 'RF',
        'randomforest': 'RF',
        'random_forest': 'RF',
        'random forest': 'RF',
        'knn': 'KNN',
        'soft_vote': 'SOFT_VOTE_EQ',
        'soft_vote_eq': 'SOFT_VOTE_EQ',
    }
    return lookup.get(text.lower(), text)


def classifier_from_setting_tag(setting_tag):
    """Extract classifier from strings such as SVM__random0_rfe_top25."""
    if setting_tag is None or (isinstance(setting_tag, float) and np.isnan(setting_tag)):
        return None
    text = str(setting_tag)
    if '__' in text:
        return text.split('__', 1)[0]
    if '/' in text:
        return text.split('/', 1)[0]
    return text.split('_', 1)[0]


def majority_algorithm_from_cv_manifest(final_selection_folder, fallback=None):
    """Use CV-selected settings only to define the algorithm label.

    If several settings were selected, the classifier appearing most often wins.
    Ties are resolved by first occurrence, which keeps the result deterministic
    and reflects the user's ordered selection list.
    """
    folder = Path(final_selection_folder)
    manifest = read_json_if_exists(folder / 'cv_final_selection_manifest.json')

    candidate_items = []
    if manifest is not None:
        for key in [
            'selected_cv_experiments',
            'selected_experiments',
            'selected_settings',
            'cv_selected_settings',
            'settings',
        ]:
            value = manifest.get(key)
            if isinstance(value, list) and len(value) > 0:
                candidate_items = value
                break

    algorithms = []
    for item in candidate_items:
        if isinstance(item, dict):
            algo = item.get('classifier') or item.get('method') or item.get('algorithm')
            if algo is None:
                algo = classifier_from_setting_tag(item.get('setting_tag') or item.get('experiment'))
        else:
            algo = classifier_from_setting_tag(item)
        if algo:
            algorithms.append(normalize_algorithm_name(algo))

    if len(algorithms) == 0:
        # Fallback to final CV metric rows if the manifest lacks selected settings.
        metric_path = folder / 'cv_final_selection_metrics.xlsx'
        if metric_path.is_file():
            df = pd.read_excel(metric_path)
            if 'is_final_selection' in df.columns:
                final_df = df[df['is_final_selection'].astype(bool)].copy()
            else:
                final_df = df.copy()
            for value in final_df.get('setting_tag', pd.Series(dtype=object)).dropna().tolist():
                algo = classifier_from_setting_tag(value)
                if algo:
                    algorithms.append(normalize_algorithm_name(algo))

    if len(algorithms) == 0 and fallback is not None:
        algorithms = [normalize_algorithm_name(fallback)]

    if len(algorithms) == 0:
        return 'NA'

    counts = Counter(algorithms)
    best_algo = None
    best_count = -1
    for algo in algorithms:
        if counts[algo] > best_count:
            best_algo = algo
            best_count = counts[algo]
    return best_algo


def read_final_selection_metric(final_selection_folder, metric_filename):
    path = Path(final_selection_folder) / metric_filename
    if not path.is_file():
        raise FileNotFoundError(f'Missing final-selection metrics file: {path}')

    df = pd.read_excel(path)
    if 'is_final_selection' in df.columns:
        final_df = df[df['is_final_selection'].astype(bool)].copy()
        if final_df.shape[0] == 0:
            raise RuntimeError(f'No is_final_selection=True row found in {path}')
        row = final_df.iloc[-1]
    else:
        row = df.iloc[-1]

    return row, path


def read_soft_vote_metric(metrics_path, dataset_key):
    path = Path(metrics_path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing soft-vote metrics file: {path}')

    df = pd.read_excel(path)
    row_df = df[df['dataset'].astype(str) == dataset_key].copy()
    if row_df.shape[0] == 0:
        raise RuntimeError(f'Dataset {dataset_key!r} not found in {path}')
    return row_df.iloc[-1], path


def format_metric(value):
    if value is None or pd.isna(value):
        return 'NA'
    return f'{float(value):.3f}'


def format_auc_ci(row):
    if 'auc_ci_low' not in row.index or 'auc_ci_high' not in row.index:
        return 'NA'
    if pd.isna(row['auc_ci_low']) or pd.isna(row['auc_ci_high']):
        return 'NA'
    return f"{float(row['auc_ci_low']):.3f}-{float(row['auc_ci_high']):.3f}"


def metrics_from_row(row):
    out = {}
    for col in metric_cols:
        if col not in row.index:
            raise KeyError(f'Missing metric column {col!r}')
        out[col] = float(row[col])
    return out

In [3]:
# ============================================================
# 3. Collect rows for the Section 1 table
# ============================================================

records = []
source_trace = []

for cohort in cohort_specs:
    for source in model_sources:
        model_name = source['model']

        if source['kind'] == 'soft_vote':
            metric_row, metric_path = read_soft_vote_metric(
                source['metrics_path'],
                cohort['soft_vote_dataset'],
            )
            algorithm = normalize_algorithm_name(source['algorithm'])
        elif source['kind'] == 'final_selection':
            metric_row, metric_path = read_final_selection_metric(
                source['folder'],
                cohort['final_selection_file'],
            )
            fallback = source.get('algorithm')
            if fallback is None and 'folder' in source:
                fallback = Path(source['folder']).name
            algorithm = majority_algorithm_from_cv_manifest(source['folder'], fallback=fallback)
        else:
            raise ValueError(f"Unknown source kind: {source['kind']}")

        metric_values = metrics_from_row(metric_row)
        record = {
            'Cohort': cohort['display'],
            'Model': model_name,
            'Algorithm': algorithm,
            '95%CI': format_auc_ci(metric_row),
            **metric_values,
        }
        records.append(record)

        source_trace.append({
            'Cohort': cohort['display'],
            'Model': model_name,
            'Algorithm': algorithm,
            'metrics_path': str(metric_path),
            'probability_column': metric_row.get('probability_column', ''),
            'setting_tag': metric_row.get('setting_tag', ''),
            'fusion_method': metric_row.get('fusion_method', metric_row.get('method', '')),
        })

performance_df = pd.DataFrame(records)
source_trace_df = pd.DataFrame(source_trace)

# Display only; intentionally do not save Excel in this notebook section.
display(performance_df)
print('\nSource trace:')
display(source_trace_df)

,Cohort,Model,Algorithm,95%CI,auc,accuracy,sensitivity,specificity
0,train,Clinical,RF,0.602-0.774,0.692997,0.707447,0.632653,0.733813
1,train,C-radiomics,SVM,0.708-0.852,0.783732,0.755319,0.734694,0.762590
2,train,H-radiomics,SVM,0.744-0.879,0.817061,0.771277,0.795918,0.762590
3,train,DL_3D,LR,0.784-0.900,0.844223,0.787234,0.795918,0.784173
4,train,fusion_soft_vote,N/A,0.852-0.940,0.899134,0.744681,0.979592,0.661871
5,train,fusion_stacking,RF,0.857-0.948,0.905741,0.851064,0.857143,0.848921
6,internal test,Clinical,RF,0.526-0.778,0.664641,0.750000,0.416667,0.861111
7,internal test,C-radiomics,SVM,0.670-0.860,0.771412,0.697917,0.916667,0.625000
8,internal test,H-radiomics,SVM,0.698-0.898,0.805556,0.760417,0.708333,0.777778
9,internal test,DL_3D,LR,0.714-0.902,0.815394,0.750000,0.791667,0.736111



Source trace:


,Cohort,Model,Algorithm,metrics_path,probability_column,setting_tag,fusion_method
0,train,Clinical,RF,/host/d/projects/Habitats/models/Prognosis/cli...,prob_final_selection,selected_settings,final_selection
1,train,C-radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/who...,prob_final_selection,selected_settings,final_selection
2,train,H-radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/hab...,prob_final_selection,selected_settings,final_selection
3,train,DL_3D,LR,/host/d/projects/Habitats/models/Prognosis/dl_...,prob_final_selection,selected_settings,final_selection
4,train,fusion_soft_vote,N/A,/host/d/projects/Habitats/models/Prognosis/fus...,prob_soft_vote,,soft_vote
5,train,fusion_stacking,RF,/host/d/projects/Habitats/models/Prognosis/fus...,prob_final_selection,selected_settings,final_selection
6,internal test,Clinical,RF,/host/d/projects/Habitats/models/Prognosis/cli...,prob_final_selection,selected_settings,final_selection
7,internal test,C-radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/who...,prob_final_selection,selected_settings,final_selection
8,internal test,H-radiomics,SVM,/host/d/projects/Habitats/models/Prognosis/hab...,prob_final_selection,selected_settings,final_selection
9,internal test,DL_3D,LR,/host/d/projects/Habitats/models/Prognosis/dl_...,prob_final_selection,selected_settings,final_selection


In [4]:
# ============================================================
# 4. Draw paper-style vector heatmap table
# ============================================================

# This plot is one vector figure. It combines text columns and colored metric cells.
plot_df = performance_df.copy()

text_columns = ['Cohort', 'Model', 'Algorithm', '95%CI']
plot_columns = text_columns + [metric_display[c] for c in metric_cols]

# Fixed model order inside each cohort block.
model_order = [source['model'] for source in model_sources]
cohort_order = [cohort['display'] for cohort in cohort_specs]

ordered_rows = []
for cohort_name in cohort_order:
    block = plot_df[plot_df['Cohort'] == cohort_name].copy()
    block['Model'] = pd.Categorical(block['Model'], categories=model_order, ordered=True)
    block = block.sort_values('Model')
    ordered_rows.append(block)
plot_df = pd.concat(ordered_rows, ignore_index=True)

# Column widths are tuned for paper-style readability.
col_widths = {
    'Cohort': 1.55,
    'Model': 2.10,
    'Algorithm': 1.45,
    '95%CI': 1.75,
    'AUC': 1.05,
    'Accuracy': 1.25,
    'Sensitivity': 1.25,
    'Specificity': 1.25,
}
widths = [col_widths[col] for col in plot_columns]
x_edges = np.concatenate([[0], np.cumsum(widths)])

def x_center(col_i):
    return (x_edges[col_i] + x_edges[col_i + 1]) / 2

n_rows = plot_df.shape[0]
header_h = 0.70
row_h = 0.56
total_w = x_edges[-1]
total_h = header_h + n_rows * row_h

fig_w = 14.8
fig_h = max(8.6, 1.4 + n_rows * 0.48)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(0, total_w)
ax.set_ylim(0, total_h)
ax.axis('off')

cmap = plt.get_cmap('RdYlBu_r')
norm = Normalize(vmin=0.45, vmax=0.95)

# Header.
header_y = total_h - header_h
for col_i, col_name in enumerate(plot_columns):
    ax.add_patch(Rectangle(
        (x_edges[col_i], header_y),
        widths[col_i],
        header_h,
        facecolor='white',
        edgecolor='black',
        linewidth=1.0,
    ))
    ax.text(
        x_center(col_i),
        header_y + header_h / 2,
        col_name,
        ha='center',
        va='center',
        fontsize=13.0,
        fontweight='bold',
    )

# Body cells.
metric_start_col = len(text_columns)
for row_i, row in plot_df.iterrows():
    y0 = total_h - header_h - (row_i + 1) * row_h

    # Text cells: cohort/model/algorithm/CI.
    text_values = ['', row['Model'], row['Algorithm'], row['95%CI']]
    for col_i, value in enumerate(text_values):
        ax.add_patch(Rectangle(
            (x_edges[col_i], y0),
            widths[col_i],
            row_h,
            facecolor='white',
            edgecolor='#444444',
            linewidth=0.45,
        ))
        if value != '':
            ax.text(
                x_center(col_i),
                y0 + row_h / 2,
                str(value),
                ha='center',
                va='center',
                fontsize=11.5,
            )

    # Metric cells.
    for j, metric in enumerate(metric_cols):
        col_i = metric_start_col + j
        value = float(row[metric])
        facecolor = cmap(norm(value))
        ax.add_patch(Rectangle(
            (x_edges[col_i], y0),
            widths[col_i],
            row_h,
            facecolor=facecolor,
            edgecolor='white',
            linewidth=0.45,
        ))

        # Darker cells need white text.
        r, g, b, _ = facecolor
        luminance = 0.299 * r + 0.587 * g + 0.114 * b
        text_color = 'white' if luminance < 0.48 else 'black'
        ax.text(
            x_center(col_i),
            y0 + row_h / 2,
            format_metric(value),
            ha='center',
            va='center',
            fontsize=11.5,
            color=text_color,
        )

# Cohort labels and block separators.
rows_per_block = len(model_order)
for block_i, cohort_name in enumerate(cohort_order):
    start_row = block_i * rows_per_block
    end_row = start_row + rows_per_block

    block_top = total_h - header_h - start_row * row_h
    block_bottom = total_h - header_h - end_row * row_h
    block_center = (block_top + block_bottom) / 2

    # Cover internal row lines in the cohort column and draw one merged-looking cohort cell.
    ax.add_patch(Rectangle(
        (x_edges[0], block_bottom),
        widths[0],
        rows_per_block * row_h,
        facecolor='white',
        edgecolor='#444444',
        linewidth=0.45,
        zorder=3,
    ))
    ax.text(
        x_center(0),
        block_center,
        cohort_name,
        ha='center',
        va='center',
        fontsize=11.8,
        fontweight='bold',
        zorder=4,
    )

    # Thick separator above each cohort block and below the final block.
    ax.plot([0, total_w], [block_top, block_top], color='black', linewidth=1.2, zorder=5)
    if block_i == len(cohort_order) - 1:
        ax.plot([0, total_w], [block_bottom, block_bottom], color='black', linewidth=1.2, zorder=5)

# Vertical boundary between text columns and heatmap metric columns.
ax.plot([x_edges[metric_start_col], x_edges[metric_start_col]], [0, total_h], color='black', linewidth=1.2, zorder=5)

# Colorbar.
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.025)
cbar.set_label('Metric Value', rotation=90, fontsize=12)
cbar.ax.tick_params(labelsize=11)

save_pdf_and_ppt_safe_svg(fig, figure_pdf_path, bbox_inches='tight')
fig.savefig(figure_eps_path, bbox_inches='tight')
plt.close(fig)

print('Saved vector figure:')
print(' ', figure_pdf_path)
print(' ', figure_eps_path)

Saved vector figure:
  /host/d/projects/Habitats/results/performance_heatmap.pdf
  /host/d/projects/Habitats/results/performance_heatmap.eps


## Section 2. ROC and DCA Curves

This section is standalone: it redefines imports, paths, model sources, and plotting functions. It reads final-selection prediction tables and generates single-panel ROC/DCA PDFs plus one combined 2 x 3 PDF.

In [5]:
# ============================================================
# Section 2. ROC and DCA curves
# This block can be run independently from Section 1.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.metrics import roc_auc_score, roc_curve

# ============================================================
# 1. Paths and font settings
# ============================================================

model_root = Path('/host/d/projects/Habitats/models/Prognosis')
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)

# Explicitly register Times New Roman from Windows fonts when available.
times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

# Cohorts and files.
cohort_specs = {
    'train': {
        'standard_prediction_file': 'cv_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'cv',
    },
    'internal_test': {
        'standard_prediction_file': 'internal_test_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'internal_test',
    },
    'external_test': {
        'standard_prediction_file': 'external_test_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'external_test',
    },
}

cohort_display = {
    'train': 'Train',
    'internal_test': 'Internal test',
    'external_test': 'External test',
}

# Model sources. Legend names are the paper-facing names requested by the user.
model_sources = [
    {
        'legend': 'Clinical',
        'kind': 'standard_final_selection',
        'folder': model_root / 'clinical' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'C-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'whole_image' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'H-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'habitats_avg' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'DL_3D',
        'kind': 'standard_final_selection',
        'folder': model_root / 'dl_3d_ml_all' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'Soft_vote',
        'kind': 'soft_vote',
        'prediction_path': model_root / 'fusion' / 'soft_vote_predictions.xlsx',
        'prob_col': 'prob_soft_vote',
    },
    {
        'legend': 'stacking',
        'kind': 'standard_final_selection',
        'folder': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF',
        'prob_col': 'prob_final_selection',
    },
]

label_col = 'Prognosis_label'

model_colors = {
    'Clinical': '#4C78A8',
    'C-radiomics': '#F58518',
    'H-radiomics': '#54A24B',
    'DL_3D': '#B279A2',
    'Soft_vote': '#E45756',
    'stacking': '#72B7B2',
}

single_roc_paths = {
    'train': results_out_dir / 'ROC_train.pdf',
    'internal_test': results_out_dir / 'ROC_internal_test.pdf',
    'external_test': results_out_dir / 'ROC_external_test.pdf',
}
single_dca_paths = {
    'train': results_out_dir / 'DCA_train.pdf',
    'internal_test': results_out_dir / 'DCA_internal_test.pdf',
    'external_test': results_out_dir / 'DCA_external_test.pdf',
}
combined_path = results_out_dir / 'ROC_DCA_combined.pdf'

print('Results directory:', results_out_dir)


Results directory: /host/d/projects/Habitats/results


In [6]:
# ============================================================
# 2. Load final-selection probabilities
# ============================================================

def load_model_prediction_for_cohort(source, cohort_key):
    """Return y_true and y_prob for one model and one cohort."""
    spec = cohort_specs[cohort_key]

    if source['kind'] == 'standard_final_selection':
        path = Path(source['folder']) / spec['standard_prediction_file']
        if not path.is_file():
            raise FileNotFoundError(f'Missing prediction file: {path}')
        df = pd.read_excel(path)
    elif source['kind'] == 'soft_vote':
        path = Path(source['prediction_path'])
        if not path.is_file():
            raise FileNotFoundError(f'Missing prediction file: {path}')
        df_all = pd.read_excel(path)
        df = df_all[df_all['dataset'].astype(str) == spec['soft_vote_dataset']].copy()
        if df.shape[0] == 0:
            raise RuntimeError(f'No soft-vote rows found for dataset={spec["soft_vote_dataset"]}')
    else:
        raise ValueError(f"Unknown model source kind: {source['kind']}")

    prob_col = source['prob_col']
    if label_col not in df.columns:
        raise KeyError(f'Missing label column {label_col} in {path}')
    if prob_col not in df.columns:
        raise KeyError(f'Missing probability column {prob_col} in {path}')

    id_cols = [col for col in ['Patient_set', 'Patient_index'] if col in df.columns]
    keep_cols = id_cols + [label_col, prob_col]
    out = df[keep_cols].copy()
    out = out.dropna(subset=[label_col, prob_col])

    # Different final-selection notebooks may save cases in different orders.
    # Sorting by case identifiers keeps the trace deterministic while preserving
    # each model's correct label/probability pairing.
    if len(id_cols) == 2:
        out = out.sort_values(id_cols).reset_index(drop=True)

    y_true = out[label_col].astype(int).to_numpy()
    y_prob = out[prob_col].astype(float).to_numpy()
    return y_true, y_prob, path


def load_all_predictions():
    all_pred = {}
    trace_rows = []
    for cohort_key in cohort_specs:
        all_pred[cohort_key] = {}
        for source in model_sources:
            y_true, y_prob, path = load_model_prediction_for_cohort(source, cohort_key)
            auc = roc_auc_score(y_true, y_prob)
            all_pred[cohort_key][source['legend']] = {
                'y_true': y_true,
                'y_prob': y_prob,
                'auc': auc,
                'path': path,
            }
            trace_rows.append({
                'cohort': cohort_key,
                'model': source['legend'],
                'n': len(y_true),
                'positive_fraction': float(np.mean(y_true)),
                'auc': auc,
                'path': str(path),
            })
    return all_pred, pd.DataFrame(trace_rows)

all_predictions, prediction_trace_df = load_all_predictions()
display(prediction_trace_df)


,cohort,model,n,positive_fraction,auc,path
0,train,Clinical,188,0.260638,0.692997,/host/d/projects/Habitats/models/Prognosis/cli...
1,train,C-radiomics,188,0.260638,0.783732,/host/d/projects/Habitats/models/Prognosis/who...
2,train,H-radiomics,188,0.260638,0.817061,/host/d/projects/Habitats/models/Prognosis/hab...
3,train,DL_3D,188,0.260638,0.844223,/host/d/projects/Habitats/models/Prognosis/dl_...
4,train,Soft_vote,188,0.260638,0.899134,/host/d/projects/Habitats/models/Prognosis/fus...
5,train,stacking,188,0.260638,0.905741,/host/d/projects/Habitats/models/Prognosis/fus...
6,internal_test,Clinical,96,0.250000,0.664641,/host/d/projects/Habitats/models/Prognosis/cli...
7,internal_test,C-radiomics,96,0.250000,0.771412,/host/d/projects/Habitats/models/Prognosis/who...
8,internal_test,H-radiomics,96,0.250000,0.805556,/host/d/projects/Habitats/models/Prognosis/hab...
9,internal_test,DL_3D,96,0.250000,0.815394,/host/d/projects/Habitats/models/Prognosis/dl_...


In [7]:
# ============================================================
# 3. ROC and DCA plotting functions
# ============================================================

def net_benefit_curve(y_true, y_prob, thresholds):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    net_benefits = []
    for threshold in thresholds:
        pred_pos = y_prob >= threshold
        tp = np.sum((pred_pos == 1) & (y_true == 1))
        fp = np.sum((pred_pos == 1) & (y_true == 0))
        nb = (tp / n) - (fp / n) * (threshold / (1.0 - threshold))
        net_benefits.append(nb)
    return np.asarray(net_benefits)


def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.35)


def plot_roc_for_cohort(cohort_key, ax=None, show_title=True, legend_loc='lower right'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    else:
        fig = ax.figure

    for source in model_sources:
        legend_name = source['legend']
        pred = all_predictions[cohort_key][legend_name]
        fpr, tpr, _ = roc_curve(pred['y_true'], pred['y_prob'])
        auc = pred['auc']
        ax.plot(
            fpr,
            tpr,
            color=model_colors[legend_name],
            linewidth=1.8,
            label=f'{legend_name}: AUC {auc:.3f}',
        )

    ax.plot([0, 1], [0, 1], color='#888888', linewidth=1.0, linestyle='--')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('1 - Specificity', fontsize=14)
    ax.set_ylabel('Sensitivity', fontsize=14)
    if show_title:
        ax.set_title(f'ROC curve: {cohort_display[cohort_key]}', fontsize=17, fontweight='bold')
    ax.legend(loc=legend_loc, fontsize=11, frameon=False, borderaxespad=0.2)
    style_axes(ax)
    return fig, ax


def plot_dca_for_cohort(cohort_key, ax=None, show_title=True, legend_loc='best'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
    else:
        fig = ax.figure

    thresholds = np.linspace(0.01, 0.99, 99)
    all_nb = []
    for source in model_sources:
        legend_name = source['legend']
        pred = all_predictions[cohort_key][legend_name]
        nb = net_benefit_curve(pred['y_true'], pred['y_prob'], thresholds)
        all_nb.append(nb)
        ax.plot(
            thresholds,
            nb,
            color=model_colors[legend_name],
            linewidth=1.8,
            label=legend_name,
        )

    all_nb = np.vstack(all_nb)
    y_min = np.nanmin(all_nb)
    y_max = np.nanmax(all_nb)
    pad = max(0.02, 0.08 * (y_max - y_min if y_max > y_min else 1.0))
    ax.set_xlim(0, 1)
    ax.set_ylim(y_min - pad, y_max + pad)
    ax.set_xlabel('Threshold probability', fontsize=14)
    ax.set_ylabel('Net benefit', fontsize=14)
    if show_title:
        ax.set_title(f'DCA curve: {cohort_display[cohort_key]}', fontsize=17, fontweight='bold')
    ax.legend(loc=legend_loc, fontsize=11, frameon=False, borderaxespad=0.2)
    style_axes(ax)
    return fig, ax

print('Plotting functions ready.')


Plotting functions ready.


In [8]:
# ============================================================
# 4. Save 6 individual PDFs and one 2 x 3 combined PDF
# ============================================================

# Individual ROC PDFs.
for cohort_key, out_path in single_roc_paths.items():
    fig, ax = plot_roc_for_cohort(cohort_key, show_title=True, legend_loc='lower right')
    fig.tight_layout()
    save_pdf_and_ppt_safe_svg(fig, out_path, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out_path)

# Individual DCA PDFs.
for cohort_key, out_path in single_dca_paths.items():
    fig, ax = plot_dca_for_cohort(cohort_key, show_title=True, legend_loc='upper right')
    fig.tight_layout()
    save_pdf_and_ppt_safe_svg(fig, out_path, bbox_inches='tight')
    plt.close(fig)
    print('Saved:', out_path)

# Combined 2 x 3 figure.
fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.7))
ordered_cohorts = ['train', 'internal_test', 'external_test']

for col_i, cohort_key in enumerate(ordered_cohorts):
    plot_roc_for_cohort(cohort_key, ax=axes[0, col_i], show_title=False, legend_loc='lower right')
    axes[0, col_i].set_title(cohort_display[cohort_key], fontsize=18, fontweight='bold')

for col_i, cohort_key in enumerate(ordered_cohorts):
    plot_dca_for_cohort(cohort_key, ax=axes[1, col_i], show_title=False, legend_loc='upper right')

# Row labels placed outside axes.

fig.tight_layout(rect=(0.01, 0.02, 1.0, 0.98), w_pad=1.2, h_pad=2.0)
save_pdf_and_ppt_safe_svg(fig, combined_path, bbox_inches='tight')
plt.close(fig)
print('Saved combined figure:', combined_path)


Saved: /host/d/projects/Habitats/results/ROC_train.pdf


Saved: /host/d/projects/Habitats/results/ROC_internal_test.pdf


Saved: /host/d/projects/Habitats/results/ROC_external_test.pdf


Saved: /host/d/projects/Habitats/results/DCA_train.pdf


Saved: /host/d/projects/Habitats/results/DCA_internal_test.pdf


Saved: /host/d/projects/Habitats/results/DCA_external_test.pdf


Saved combined figure: /host/d/projects/Habitats/results/ROC_DCA_combined.pdf


## Section 3. DeLong Test Heatmaps

This section is standalone: it redefines imports, paths, model sources, the DeLong implementation, and plotting functions. It computes pairwise DeLong p-values for AUC differences among final-selection models in the train, internal test, and external test cohorts.

In [9]:
# ============================================================
# Section 3. DeLong test heatmaps
# This block can be run independently from Sections 1 and 2.
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats
from sklearn.metrics import roc_auc_score

# ============================================================
# 1. Paths, model sources, and font settings
# ============================================================

model_root = Path('/host/d/projects/Habitats/models/Prognosis')
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)

times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

cohort_specs = {
    'train': {
        'display': 'Train',
        'standard_prediction_file': 'cv_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'cv',
        'single_pdf': results_out_dir / 'DeLong_train.pdf',
    },
    'internal_test': {
        'display': 'Internal test',
        'standard_prediction_file': 'internal_test_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'internal_test',
        'single_pdf': results_out_dir / 'DeLong_internal_test.pdf',
    },
    'external_test': {
        'display': 'External test',
        'standard_prediction_file': 'external_test_final_selection_predictions.xlsx',
        'soft_vote_dataset': 'external_test',
        'single_pdf': results_out_dir / 'DeLong_external_test.pdf',
    },
}

combined_pdf_path = results_out_dir / 'DeLong_combined.pdf'

model_sources = [
    {
        'legend': 'Clinical',
        'kind': 'standard_final_selection',
        'folder': model_root / 'clinical' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'C-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'whole_image' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'H-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'habitats_avg' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'DL_3D',
        'kind': 'standard_final_selection',
        'folder': model_root / 'dl_3d_ml_all' / 'final_selections',
        'prob_col': 'prob_final_selection',
    },
    {
        'legend': 'Soft_vote',
        'kind': 'soft_vote',
        'prediction_path': model_root / 'fusion' / 'soft_vote_predictions.xlsx',
        'prob_col': 'prob_soft_vote',
    },
    {
        'legend': 'stacking',
        'kind': 'standard_final_selection',
        'folder': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF',
        'prob_col': 'prob_final_selection',
    },
]

model_names = [source['legend'] for source in model_sources]
label_col = 'Prognosis_label'
id_cols = ['Patient_set', 'Patient_index']

print('Section 3 output directory:', results_out_dir)
print('Models:', model_names)


Section 3 output directory: /host/d/projects/Habitats/results
Models: ['Clinical', 'C-radiomics', 'H-radiomics', 'DL_3D', 'Soft_vote', 'stacking']


In [10]:
# ============================================================
# 2. Load and merge final-selection probabilities by case ID
# ============================================================

def load_prediction_table_for_source(source, cohort_key):
    spec = cohort_specs[cohort_key]
    if source['kind'] == 'standard_final_selection':
        path = Path(source['folder']) / spec['standard_prediction_file']
        if not path.is_file():
            raise FileNotFoundError(f'Missing prediction file: {path}')
        df = pd.read_excel(path)
    elif source['kind'] == 'soft_vote':
        path = Path(source['prediction_path'])
        if not path.is_file():
            raise FileNotFoundError(f'Missing prediction file: {path}')
        df_all = pd.read_excel(path)
        df = df_all[df_all['dataset'].astype(str) == spec['soft_vote_dataset']].copy()
        if df.shape[0] == 0:
            raise RuntimeError(f'No rows found in {path} for dataset={spec["soft_vote_dataset"]}')
    else:
        raise ValueError(f"Unknown source kind: {source['kind']}")

    prob_col = source['prob_col']
    required_cols = id_cols + [label_col, prob_col]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f'Missing columns {missing_cols} in {path}')

    out = df[required_cols].copy()
    out = out.dropna(subset=[label_col, prob_col])
    out[label_col] = out[label_col].astype(int)
    out[prob_col] = out[prob_col].astype(float)
    out = out.rename(columns={prob_col: source['legend']})
    return out, path


def load_merged_predictions_for_cohort(cohort_key):
    merged_df = None
    source_trace = []

    for source in model_sources:
        table, path = load_prediction_table_for_source(source, cohort_key)
        source_trace.append({
            'cohort': cohort_key,
            'model': source['legend'],
            'n': table.shape[0],
            'path': str(path),
        })

        if merged_df is None:
            merged_df = table.copy()
        else:
            merged_df = merged_df.merge(
                table,
                on=id_cols,
                how='inner',
                suffixes=('', '_new'),
            )
            new_label_col = f'{label_col}_new'
            if new_label_col in merged_df.columns:
                if not (merged_df[label_col].astype(int) == merged_df[new_label_col].astype(int)).all():
                    raise RuntimeError(f'Label mismatch after merging model={source["legend"]}, cohort={cohort_key}')
                merged_df = merged_df.drop(columns=[new_label_col])

    if merged_df is None or merged_df.shape[0] == 0:
        raise RuntimeError(f'No merged predictions for cohort={cohort_key}')

    merged_df = merged_df.sort_values(id_cols).reset_index(drop=True)
    return merged_df, pd.DataFrame(source_trace)

merged_predictions = {}
trace_tables = []
for cohort_key in cohort_specs:
    merged_df, trace_df = load_merged_predictions_for_cohort(cohort_key)
    merged_predictions[cohort_key] = merged_df
    trace_tables.append(trace_df)

prediction_source_trace_df = pd.concat(trace_tables, ignore_index=True)
display(prediction_source_trace_df)

auc_trace = []
for cohort_key, df in merged_predictions.items():
    y_true = df[label_col].astype(int).to_numpy()
    for model_name in model_names:
        auc_trace.append({
            'cohort': cohort_key,
            'model': model_name,
            'n_merged': df.shape[0],
            'positive_fraction': float(np.mean(y_true)),
            'auc': roc_auc_score(y_true, df[model_name].astype(float).to_numpy()),
        })
auc_trace_df = pd.DataFrame(auc_trace)
display(auc_trace_df)


,cohort,model,n,path
0,train,Clinical,188,/host/d/projects/Habitats/models/Prognosis/cli...
1,train,C-radiomics,188,/host/d/projects/Habitats/models/Prognosis/who...
2,train,H-radiomics,188,/host/d/projects/Habitats/models/Prognosis/hab...
3,train,DL_3D,188,/host/d/projects/Habitats/models/Prognosis/dl_...
4,train,Soft_vote,188,/host/d/projects/Habitats/models/Prognosis/fus...
5,train,stacking,188,/host/d/projects/Habitats/models/Prognosis/fus...
6,internal_test,Clinical,96,/host/d/projects/Habitats/models/Prognosis/cli...
7,internal_test,C-radiomics,96,/host/d/projects/Habitats/models/Prognosis/who...
8,internal_test,H-radiomics,96,/host/d/projects/Habitats/models/Prognosis/hab...
9,internal_test,DL_3D,96,/host/d/projects/Habitats/models/Prognosis/dl_...


,cohort,model,n_merged,positive_fraction,auc
0,train,Clinical,188,0.260638,0.692997
1,train,C-radiomics,188,0.260638,0.783732
2,train,H-radiomics,188,0.260638,0.817061
3,train,DL_3D,188,0.260638,0.844223
4,train,Soft_vote,188,0.260638,0.899134
5,train,stacking,188,0.260638,0.905741
6,internal_test,Clinical,96,0.250000,0.664641
7,internal_test,C-radiomics,96,0.250000,0.771412
8,internal_test,H-radiomics,96,0.250000,0.805556
9,internal_test,DL_3D,96,0.250000,0.815394


In [11]:
# ============================================================
# 3. DeLong implementation
# ============================================================

# This implementation follows the midrank-based fast DeLong method.
# It returns a two-sided p-value for H0: AUC_model_1 == AUC_model_2.

def compute_midrank(x):
    x = np.asarray(x)
    order = np.argsort(x)
    sorted_x = x[order]
    n = len(x)
    midranks = np.zeros(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j < n and sorted_x[j] == sorted_x[i]:
            j += 1
        midrank_value = 0.5 * (i + j - 1) + 1.0
        midranks[i:j] = midrank_value
        i = j
    out = np.empty(n, dtype=float)
    out[order] = midranks
    return out


def fast_delong(predictions_sorted_transposed, label_1_count):
    m = int(label_1_count)
    n = predictions_sorted_transposed.shape[1] - m
    if m <= 0 or n <= 0:
        raise ValueError('DeLong requires at least one positive and one negative case.')

    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty((k, m), dtype=float)
    ty = np.empty((k, n), dtype=float)
    tz = np.empty((k, m + n), dtype=float)

    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])

    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delong_cov = sx / m + sy / n
    return aucs, delong_cov


def delong_roc_test(y_true, pred_one, pred_two):
    y_true = np.asarray(y_true).astype(int)
    pred_one = np.asarray(pred_one).astype(float)
    pred_two = np.asarray(pred_two).astype(float)

    if not set(np.unique(y_true)).issubset({0, 1}):
        raise ValueError('y_true must be binary 0/1.')

    order = np.argsort(-y_true)
    label_1_count = int(np.sum(y_true == 1))
    predictions = np.vstack([pred_one, pred_two])[:, order]
    aucs, covariance = fast_delong(predictions, label_1_count)

    # Difference variance. Covariance can be scalar-like in degenerate cases,
    # so force it to a 2x2 array.
    covariance = np.atleast_2d(covariance)
    contrast = np.array([1.0, -1.0])
    diff = aucs[0] - aucs[1]
    var = float(contrast @ covariance @ contrast.T)

    if var <= 0 or np.isclose(var, 0):
        return 1.0 if np.isclose(diff, 0) else 0.0

    z = abs(diff) / np.sqrt(var)
    p_value = 2.0 * (1.0 - stats.norm.cdf(z))
    return float(np.clip(p_value, 0.0, 1.0))


def compute_delong_matrix(cohort_key):
    df = merged_predictions[cohort_key]
    y_true = df[label_col].astype(int).to_numpy()
    n_models = len(model_names)
    matrix = np.ones((n_models, n_models), dtype=float)

    for row_i in range(n_models):
        for col_i in range(n_models):
            if row_i > col_i:
                p = delong_roc_test(
                    y_true,
                    df[model_names[row_i]].astype(float).to_numpy(),
                    df[model_names[col_i]].astype(float).to_numpy(),
                )
                matrix[row_i, col_i] = p
            else:
                matrix[row_i, col_i] = 1.0

    return pd.DataFrame(matrix, index=model_names, columns=model_names)

delong_matrices = {}
for cohort_key in cohort_specs:
    delong_matrices[cohort_key] = compute_delong_matrix(cohort_key)
    print('\n', cohort_key)
    display(delong_matrices[cohort_key])



 train


,Clinical,C-radiomics,H-radiomics,DL_3D,Soft_vote,stacking
Clinical,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.0
C-radiomics,1.147565e-01,1.000000,1.000000,1.000000,1.000000,1.0
H-radiomics,1.883037e-02,0.434013,1.000000,1.000000,1.000000,1.0
DL_3D,2.976961e-03,0.183523,0.545786,1.000000,1.000000,1.0
Soft_vote,7.740038e-07,0.000574,0.014592,0.004909,1.000000,1.0
stacking,7.198150e-06,0.000030,0.001907,0.021962,0.648248,1.0



 internal_test


,Clinical,C-radiomics,H-radiomics,DL_3D,Soft_vote,stacking
Clinical,1.000000,1.000000,1.000000,1.000000,1.00000,1.0
C-radiomics,0.183222,1.000000,1.000000,1.000000,1.00000,1.0
H-radiomics,0.115514,0.562628,1.000000,1.000000,1.00000,1.0
DL_3D,0.070257,0.500027,0.846911,1.000000,1.00000,1.0
Soft_vote,0.000737,0.043297,0.185007,0.132103,1.00000,1.0
stacking,0.008916,0.030397,0.053322,0.096913,0.94733,1.0



 external_test


,Clinical,C-radiomics,H-radiomics,DL_3D,Soft_vote,stacking
Clinical,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
C-radiomics,0.152946,1.000000,1.000000,1.000000,1.000000,1.0
H-radiomics,0.089224,0.751838,1.000000,1.000000,1.000000,1.0
DL_3D,0.093343,0.768893,0.964437,1.000000,1.000000,1.0
Soft_vote,0.000767,0.339765,0.579722,0.458608,1.000000,1.0
stacking,0.003507,0.107750,0.291738,0.167890,0.347822,1.0


In [12]:
# ============================================================
# 4. Plot DeLong heatmaps: three singles and one combined 1 x 3
# ============================================================

cmap = plt.get_cmap('viridis')

def format_p_value_for_cell(p):
    if pd.isna(p):
        return 'NA'
    return f'{float(p):.3f}'


def plot_delong_heatmap(cohort_key, ax=None, show_colorbar=True, colorbar_ax=None, title=None):
    matrix_df = delong_matrices[cohort_key]
    values = matrix_df.to_numpy(dtype=float)

    if ax is None:
        fig, ax = plt.subplots(figsize=(5.4, 4.8))
    else:
        fig = ax.figure

    im = ax.imshow(values, cmap=cmap, vmin=0.0, vmax=1.0, aspect='equal')

    ax.set_xticks(np.arange(len(model_names)))
    ax.set_yticks(np.arange(len(model_names)))
    ax.set_xticklabels(model_names, rotation=90, fontsize=9)
    ax.set_yticklabels(model_names, fontsize=9)
    ax.tick_params(length=0)

    if title is None:
        title = f'{cohort_specs[cohort_key]["display"]} DeLong (p-value)'
    ax.set_title(title, fontsize=11)

    # Cell labels. White text on dark cells, black text on light cells.
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            p = values[i, j]
            text_color = 'black' if p >= 0.85 else 'white'
            ax.text(j, i, format_p_value_for_cell(p), ha='center', va='center', fontsize=8, color=text_color)

    # Thin grid lines.
    ax.set_xticks(np.arange(-0.5, len(model_names), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(model_names), 1), minor=True)
    ax.grid(which='minor', color='white', linestyle='-', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    if show_colorbar:
        cbar = fig.colorbar(im, ax=ax if colorbar_ax is None else None, cax=colorbar_ax, fraction=0.046, pad=0.04)
        cbar.set_label('p-value', fontsize=10)
        cbar.ax.tick_params(labelsize=9)

    return fig, ax, im

# Individual PDFs, each with its own colorbar.
for cohort_key, spec in cohort_specs.items():
    fig, ax, im = plot_delong_heatmap(cohort_key, show_colorbar=True)
    fig.tight_layout()
    save_pdf_and_ppt_safe_svg(fig, spec['single_pdf'], bbox_inches='tight')
    plt.close(fig)
    print('Saved:', spec['single_pdf'])

# Combined 1 x 3 PDF with one shared colorbar.
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.9))
combined_order = ['train', 'internal_test', 'external_test']
ims = []
for panel_i, cohort_key in enumerate(combined_order):
    _, _, im = plot_delong_heatmap(
        cohort_key,
        ax=axes[panel_i],
        show_colorbar=False,
        title=f'{chr(65 + panel_i)}. {cohort_specs[cohort_key]["display"]}',
    )
    ims.append(im)

cbar = fig.colorbar(ims[-1], ax=axes.ravel().tolist(), fraction=0.025, pad=0.025)
cbar.set_label('p-value', fontsize=11)
cbar.ax.tick_params(labelsize=9)
save_pdf_and_ppt_safe_svg(fig, combined_pdf_path, bbox_inches='tight')
plt.close(fig)
print('Saved combined DeLong heatmap:', combined_pdf_path)


Saved: /host/d/projects/Habitats/results/DeLong_train.pdf


Saved: /host/d/projects/Habitats/results/DeLong_internal_test.pdf


Saved: /host/d/projects/Habitats/results/DeLong_external_test.pdf


Saved combined DeLong heatmap: /host/d/projects/Habitats/results/DeLong_combined.pdf
